In [6]:
## TODO: en meget vigtig ting du ikke har kigget på endnu er chamber-fiber_idx korrespondances, dette kunne være make-or-break

In [ ]:
# === BOILERPLATE: Load Sessions ===
import sys, pickle, shutil
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import gc

# Configure Plotting
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.2)

# Repo Discovery
start = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd().resolve()
for p in (start, *start.parents):
    if p.name == "syllable_generation":
        REPO_ROOT = p
        break
else:
    REPO_ROOT = Path("/home/fsp585/Desktop/FiberphotometryCode/syllable_generation")

PKG_PARENT = REPO_ROOT.parent
if str(PKG_PARENT) not in sys.path:
    sys.path.insert(0, str(PKG_PARENT))

# Load Sessions
PKL = REPO_ROOT / "loaders" / "jonathan_ifdn_sessions_w_sylls.pkl"
if not PKL.exists():
    loaders_dir = REPO_ROOT / "loaders"
    candidates = list(loaders_dir.glob("*.pkl"))
    if not candidates:
        raise FileNotFoundError("No pickle found.")
    PKL = max(candidates, key=lambda p: p.stat().st_mtime)

print(f"Loading sessions from {PKL}...")
with PKL.open("rb") as f:
    sessions = pickle.load(f)

# Quick filter
sessions = [
    s for s in sessions
    if hasattr(s, "all_data_df") and isinstance(s.all_data_df, pd.DataFrame) and not s.all_data_df.empty
]
print(f"Usable sessions: {len(sessions)}")

Loading sessions from /home/fsp585/Desktop/FiberphotometryCode/syllable_generation/loaders/jonathan_ifdn_sessions_w_sylls_datta.pkl...
Usable sessions: 19


In [8]:
# === CONFIGURATION KNOBS ===

# 1. Output Settings
OUTPUT_ROOT = Path("sanity_check_plots") 
CLEAR_OLD_PLOTS = True 

# 2. Time Window
FS = 80.0
PRE_S = 5.0   
POST_S = 5.0  

# 3. Normalization
BASELINE_MODE = "point_zero" 
BASELINE_WIN  = (-1.0, -0.5) 

# 4. Filters
MIN_BOUT_DURATION = 0.0 
MIN_INSTANCES = 5         

# 5. Plot Types
PLOT_TYPES = {
    "Standard Z": "zscore",
    "Datta (RANSAC)": "zscore_datta"
}

# NEW: Color Palettes for channels
CHANNEL_COLORS = {
    "Green": {
        "Standard Z": "tab:green",
        "Datta (RANSAC)": "teal"
    },
    "Red": {
        "Standard Z": "tab:red",
        "Datta (RANSAC)": "maroon"
    }
}

In [9]:
# === HELPER FUNCTIONS ===

def get_channel_columns(session, area):
    """
    Returns a dict mapping 'Green'/'Red' to their specific columns.
    Example output:
    {
      'Green': {'Standard Z': 'signal_0_zscore', ...},
      'Red':   {'Standard Z': 'signal_1_zscore', ...}
    }
    """
    meta = session.metadata
    signal_meta = meta.get("signal_meta", {}) or {}
    
    results = {"Green": {}, "Red": {}}
    cols = session.all_data_df.columns

    for key, info in signal_meta.items():
        # Match Brain Region
        if str(info.get("brain_reg", "")).lower() == area.lower():
            
            # Determine Channel Color
            letter = str(info.get("orig_letter", "")).upper()
            if letter == "G":
                chan = "Green"
            elif letter == "R":
                chan = "Red"
            else:
                continue # Skip unknown fibers

            # Check which columns exist for this signal
            col_std = f"{key}_zscore"
            col_datta = f"{key}_zscore_datta"

            if col_std in cols:
                results[chan]["Standard Z"] = col_std
            if col_datta in cols:
                results[chan]["Datta (RANSAC)"] = col_datta

    return results

def detect_onsets_by_syllable(df, min_dur_s=0.15, fs=80.0):
    """Returns dict: { syllable_id: [frame_indices] }"""
    syl = df["syllable"].to_numpy()
    
    # Find changes
    is_change = np.concatenate(([1], syl[1:] != syl[:-1], [1]))
    change_idx = np.flatnonzero(is_change)
    
    starts = change_idx[:-1]
    ends = change_idx[1:]
    durations = ends - starts
    
    min_frames = int(min_dur_s * fs)
    valid = durations >= min_frames
    starts = starts[valid]
    
    syl_vals = syl[starts]
    unique_syls = np.unique(syl_vals)
    unique_syls = unique_syls[~np.isnan(unique_syls.astype(float))]
    
    onsets_dict = {}
    for s_id in unique_syls:
        s_id_int = int(s_id)
        idxs = starts[syl_vals == s_id]
        onsets_dict[s_id_int] = idxs
        
    return onsets_dict

def extract_traces(df, onsets, col_name, pre_f, post_f):
    if not col_name or col_name not in df.columns:
        return None
    
    data = df[col_name].to_numpy()
    n_frames = len(data)
    
    valid_mask = (onsets + pre_f >= 0) & (onsets + post_f < n_frames)
    valid_onsets = onsets[valid_mask]
    
    if len(valid_onsets) == 0:
        return None
    
    window_offsets = np.arange(pre_f, post_f + 1)
    idx_matrix = valid_onsets[:, None] + window_offsets[None, :]
    
    return data[idx_matrix]

def calculate_stats(traces, baseline_mode="point_zero", fs=80.0, pre_f=0):
    if baseline_mode == "point_zero":
        zero_idx = abs(pre_f)
        baseline_vals = traces[:, zero_idx][:, None] 
        traces_corrected = traces - baseline_vals
        
    elif baseline_mode == "window":
        w_start_idx = int((BASELINE_WIN[0] - (-abs(pre_f)/fs)) * fs)
        w_end_idx   = int((BASELINE_WIN[1] - (-abs(pre_f)/fs)) * fs)
        w_start_idx = max(0, w_start_idx)
        w_end_idx   = min(traces.shape[1], w_end_idx)
        
        if w_end_idx > w_start_idx:
            baseline_vals = np.nanmean(traces[:, w_start_idx:w_end_idx], axis=1, keepdims=True)
            traces_corrected = traces - baseline_vals
        else:
            traces_corrected = traces 
    else:
        traces_corrected = traces

    mu = np.nanmean(traces_corrected, axis=0)
    n = traces_corrected.shape[0]
    sem = np.nanstd(traces_corrected, axis=0) / np.sqrt(n)
    return mu, mu - 1.96 * sem, mu + 1.96 * sem

In [10]:
# === MAIN EXECUTION ===

if CLEAR_OLD_PLOTS and OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(exist_ok=True)

pre_f = int(round(-abs(PRE_S) * FS))
post_f = int(round(abs(POST_S) * FS))
time_axis = np.linspace(pre_f/FS, post_f/FS, (post_f - pre_f + 1))

genotypes = ["WT", "KI"]
areas = ["DLS", "DMS", "VS"]
# We iterate channels explicitly later

for geno in genotypes:
    for area in areas:
        print(f"\n=== Analyzing {geno} - {area} ===")
        
        # Container: { syllable_id: { 'Green': {type: []}, 'Red': {type: []} } }
        syllable_data = {}
        
        # A. Loop Sessions
        target_sessions = [
            s for s in sessions 
            if (getattr(s, "metadata", {}) or {}).get("genotype", "UNK") == geno
        ]
        
        for sess in tqdm(target_sessions, desc="Sessions", leave=False):
            # Get columns for both channels
            chan_map = get_channel_columns(sess, area) # {'Green': {...}, 'Red': {...}}
            
            # Detect Onsets 
            onsets_dict = detect_onsets_by_syllable(sess.all_data_df, min_dur_s=MIN_BOUT_DURATION, fs=FS)
            
            # Extract traces
            for syl_id, indices in onsets_dict.items():
                if syl_id not in syllable_data:
                    syllable_data[syl_id] = {
                        "Green": {k: [] for k in PLOT_TYPES},
                        "Red":   {k: [] for k in PLOT_TYPES}
                    }
                
                # Loop Extract: Green then Red
                for chan_name, col_dict in chan_map.items():
                    for plot_label, col_name in col_dict.items():
                        traces = extract_traces(sess.all_data_df, indices, col_name, pre_f, post_f)
                        if traces is not None:
                            syllable_data[syl_id][chan_name][plot_label].append(traces)

        # B. Plotting per Syllable (Split by Channel)
        print(f"  Processing {len(syllable_data)} syllables...")
        
        for syl_id in sorted(syllable_data.keys()):
            
            # Loop over channels to generate TWO pngs per syllable
            for channel in ["Green", "Red"]:
                
                data_map = syllable_data[syl_id][channel]
                
                # Check if we have enough data for this specific channel
                max_n = 0
                for k in PLOT_TYPES.keys():
                    n = sum(len(c) for c in data_map.get(k, []))
                    if n > max_n: max_n = n
                
                if max_n < MIN_INSTANCES:
                    continue # Skip if this channel is empty or low N
                
                # Setup Directory: output/WT/DLS/Green/syllable_1.png
                save_dir = OUTPUT_ROOT / geno / area / channel
                save_dir.mkdir(parents=True, exist_ok=True)

                # Setup Figure
                n_cols = len(PLOT_TYPES)
                fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 4), sharex=True)
                if n_cols == 1: axes = [axes]
                
                # Plot
                for ax, (label, _) in zip(axes, PLOT_TYPES.items()):
                    chunks = data_map.get(label, [])
                    
                    # Standard setup
                    ax.axvline(0, color="k", linestyle="--", alpha=0.5)
                    ax.axhline(0, color="k", linestyle="-", linewidth=0.5)
                    ax.set_title(label)
                    ax.set_xlabel("Time from Onset (s)")
                    ax.set_xlim(PRE_S * -1, POST_S)
                    
                    if chunks:
                        full_matrix = np.vstack(chunks)
                        mu, lo, hi = calculate_stats(full_matrix, BASELINE_MODE, FS, pre_f)
                        
                        # Pick color based on Channel + Type
                        c = CHANNEL_COLORS[channel].get(label, "black")
                        
                        ax.plot(time_axis, mu, color=c, label=f"n={len(full_matrix)}", lw=2)
                        ax.fill_between(time_axis, lo, hi, color=c, alpha=0.2)
                        ax.legend(fontsize="small", loc="best")
                    else:
                        ax.text(0.5, 0.5, "No Data", ha='center', transform=ax.transAxes)

                axes[0].set_ylabel(f"{channel} Z-Score")
                fig.suptitle(f"Syllable {syl_id} | {geno} {area} | {channel} Channel", y=1.02)
                sns.despine()
                plt.tight_layout()
                
                plt.savefig(save_dir / f"syllable_{syl_id}.png", dpi=100)
                plt.close(fig)
        
        del syllable_data
        gc.collect()

print("\nDone! Check 'sanity_check_plots' for Green/Red folders.")


=== Analyzing WT - DLS ===


Sessions:   0%|          | 0/7 [00:00<?, ?it/s]

  Processing 94 syllables...

=== Analyzing WT - DMS ===


Sessions:   0%|          | 0/7 [00:00<?, ?it/s]

  Processing 94 syllables...

=== Analyzing WT - VS ===


...

  Processing 92 syllables...

=== Analyzing KI - DMS ===


Sessions:   0%|          | 0/8 [00:00<?, ?it/s]

  Processing 92 syllables...

=== Analyzing KI - VS ===


Sessions:   0%|          | 0/8 [00:00<?, ?it/s]

  Processing 92 syllables...

Done! Check 'sanity_check_plots' for Green/Red folders.
